In [ ]:
import re
import pandas as pd 

from pathlib import Path


pd.set_option("display.max_colwidth", None)

RAW_FILE = Path("../data/raw/pao_metadata/Book1.xlsx")
REVIEW_FILE = Path("../data/interim/pao_label_review.csv")
OUTPUT_FILE = Path("../data/processed/pao_metadata_v1.parquet")

print(f"raw file", RAW_FILE.exists(),  "\nreview file: ",  REVIEW_FILE.exists(), "\nOutput File: ", OUTPUT_FILE.exists())

raw file True 
review file:  True 
Output File:  False


In [15]:
HEADER_ROWS = {
    "COMMUNICATION": 2,
    "CLEARANCE": 0,
    "AOsMEMOs": 2,
    "LEAVES": 0,
}

sheets={}


for name, header in HEADER_ROWS.items():
    df = pd.read_excel(RAW_FILE, sheet_name=name, header=header)
    df.columns = df.columns.astype(str).str.strip()
    sheets[name] = df
    print(f"{name:15} rows={df.shape[0]:5} columns={df.shape[1]}")


COMMUNICATION   rows= 2285 columns=12
CLEARANCE       rows=  702 columns=16
AOsMEMOs        rows=   14 columns=6
LEAVES          rows=  657 columns=9


In [30]:
for name, df in sheets.items():
    print(f"\n---{name}---")
    print(df.columns.tolist())

for name, df in sheets.items():
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]

    if not unnamed:
        print(f"{name:15} no Unnamed Columns")
        continue

    print(f"\n{name}:")
    for col in unnamed:
        filled=df[col].notna().sum()
        print(f" {col:12} values: {filled}")
        if filled > 0:
            display(df.loc[df[col].notna(), [col]].head(5))


comm = sheets["COMMUNICATION"]
rows = comm["Unnamed: 10"].notna()


display(comm.loc[rows, ["SUBJECT","DESCRIPTION", "Unnamed: 10"]])

subject = comm["SUBJECT"].fillna("").astype(str).str.strip().str.upper()
desc    = comm["DESCRIPTION"].fillna("").astype(str).str.strip().str.upper()

same       = subject == desc
desc_empty = desc == ""

# subject_in_desc = [s in d for s, d in zip(subject, desc)]
subject_in_desc = [s != "" and s in d for s, d in zip(subject, desc)]

print("rows:                        ", len(comm))
print("DESCRIPTION same as SUBJECT: ", same.sum())
print("DESCRIPTION empty:           ", desc_empty.sum())
print("SUBJECT contained in DESC:   ", sum(subject_in_desc))


---COMMUNICATION---
['DATE', 'CONTROL#', 'SUBJECT', 'DESCRIPTION', 'TYPE', 'OFFICE/PERSON FROM', 'OFFICE/UNIT CONCERNED', 'REMARKS', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

---CLEARANCE---
['DATE', 'CONTROL#', 'FULL N AME', 'OFFICE', 'PURPOSE', 'REMARKS', 'RELEASING DATE', 'C/O', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15']

---AOsMEMOs---
['DATE RECEIVED', 'DOCUMENT DATE', 'AO/MEMO/EO #', 'SUBJECT', 'OFFICE/UNIT/PERSON FROM', 'REMARKS']

---LEAVES---
['DATE FILED', 'TRANSMITAL DATE', 'CONTROL#', 'NAME', 'TYPE OF LEAVE', 'REMARKS', 'DAY/S APPLIED', 'DATE RECEIVED', 'STATUS']

COMMUNICATION:
 Unnamed: 9   values: 0
 Unnamed: 10  values: 3


,Unnamed: 10
446,"THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,Orientation on Konsulta Package: An Initiative Towards a Functional Health Care Provider Network (HCPN)


 Unnamed: 11  values: 3


,Unnamed: 11
446,"THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN)



CLEARANCE:
 Unnamed: 9   values: 0
 Unnamed: 10  values: 0
 Unnamed: 11  values: 0
 Unnamed: 12  values: 0
 Unnamed: 13  values: 0
 Unnamed: 14  values: 0
 Unnamed: 15  values: 1


,Unnamed: 15
272,


AOsMEMOs        no Unnamed Columns
LEAVES          no Unnamed Columns


,SUBJECT,DESCRIPTION,Unnamed: 10
446,"THE AMOUNT THREE HUNDRED FORTY NINE MILLION SIX HUNDRED FIFTY THREE THOUSAND TWO HUNDRED SEVENTEEN PESOS ONLY (349,653,217.00) HAS BEEN CREDITED TO YOUR ACCOUNT ON JANUARY 31, 2025.","THE AMOUNT THREE HUNDRED FORTY NINE MILLION SIX HUNDRED FIFTY THREE THOUSAND TWO HUNDRED SEVENTEEN PESOS ONLY (349,653,217.00) HAS BEEN CREDITED TO YOUR ACCOUNT ON JANUARY 31, 2025.","THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"INVITATION - 2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY.","INVITATION - 2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY.","2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN),ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN),Orientation on Konsulta Package: An Initiative Towards a Functional Health Care Provider Network (HCPN)


rows:                         2285
DESCRIPTION same as SUBJECT:  1604
DESCRIPTION empty:            0
SUBJECT contained in DESC:    1722


In [ ]:
def join_unique(values):
    result = ""
    for value in values:
        value = str(value).strip()
        if value == "" or value.lower() == "nan":
            continue
        if value.upper() in result.upper():
            continue
        if result.upper() in value.upper():
            result=value
        else:
            result=(result + " " + value).strip()
    return result